# Ingest circuits.csv file
1. Read the file using spark dataframe reader API
1. Add Metadata Columns 
    - Source File
    - Ingestion Timestamp
1. Write to bronze delta table   

![Incremental Data Processing](../common/images/formula1-incremental-data-processing.png "Incremental Data Processing")

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../common/env_config

In [0]:
%run ../common/bronze_helpers

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/circuits.csv"
table_name = f"{catalog_name}.{bronze_schema}.circuits"

#### Step 1 - Read the CSV file using the dataframe reader API

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

circuits_schema = StructType([
    StructField('circuitId',   StringType()),
    StructField("url",         StringType()),
    StructField("circuitName", StringType()),
    StructField("lat",         DoubleType()),
    StructField("long",        DoubleType()),
    StructField("locality",    StringType()),
    StructField("country",     StringType())
])

In [0]:
circuits_df = (
    spark.read
         .format('csv')
         .option('header', 'true')
#         .option('inferSchema', 'true')
         .option('mode', 'FAILFAST')
         .schema(circuits_schema)
         .load(source_file)
)

In [0]:
display(circuits_df)

#### Step 2 - Add Metadata Columns
- Source File
- Ingestion Timestamp

In [0]:
circuits_final_df = add_ingestion_metadata(circuits_df)

In [0]:
display(circuits_final_df)

#### Step 3 - Write to bronze delta table

In [0]:
# circuits_final_df = circuits_final_df.withColumn("batch_id", F.lit(v_batch_id))

In [0]:
# (
#     circuits_final_df
#         .write
#         .format('delta')
#         .mode('overwrite')
#         .option('replaceWhere', f"batch_id = '{v_batch_id}'")
#         .option('mergeSchema', 'true')
#         .saveAsTable(table_name)
# )

In [0]:
write_to_bronze (
    input_df = circuits_final_df,
    target_table = table_name,
    batch_id = v_batch_id
)

In [0]:
%sql
SELECT * FROM formula_one_incr.bronze.circuits

In [0]:
display(spark.table(table_name))

In [0]:
# %sql
# TRUNCATE TABLE formula_one_incr.bronze.circuits 